# Collections and Objects: Organising Data, Then Behaviour

Single variables stop scaling fast — real programs juggle *hundreds* of temperatures, names, and scores. Python's built-in **collections** (lists, dicts, sets, tuples) organise data; **classes** go one step further and bundle data *together with the functions that operate on it*. This lab builds from one to the other, ending with a mini gradebook that combines both.

**How to use this notebook:** run cells top to bottom with `Shift+Enter` — later cells reuse classes defined earlier. Restart and re-run all if you lose track of what's defined.

## Lists: ordered, changeable sequences

A **list** holds items in order. You can index into it (counting from 0!), slice it, append to it, and change it in place — lists are **mutable** (changeable after creation).

In [ ]:
temperatures = [18.5, 21.0, 19.2, 23.8, 17.1]

print(temperatures[0], temperatures[-1])   # first and last (negative = from the end)
print(temperatures[1:3])                   # slice: items 1 and 2 (3 excluded)
print(len(temperatures))

temperatures.append(20.4)                  # grow at the end
temperatures[0] = 18.9                     # replace in place
print(temperatures)
print(min(temperatures), max(temperatures), sum(temperatures))

## The aliasing trap

Now the single most common beginner bug with lists. A variable does not *contain* a list — it holds a *reference* to one (think: a sticky label on a box, not the box itself). Assigning `b = a` copies the **label**, not the box. Predict what the next cell prints before running it.

In [ ]:
a = [1, 2, 3]
b = a               # b is now ANOTHER NAME for the same list
b.append(99)

print("a =", a)     # surprise?
print("b =", b)
print("same object?", a is b, "- id(a):", id(a), " id(b):", id(b))

`a` changed even though we only touched `b` — because there is only **one list** with two labels on it. The `is` operator and matching `id()` numbers (an object's unique identity, roughly its address) confirm it. This situation is called **aliasing**.

When you want an actual second box, ask for a copy explicitly:

In [ ]:
a = [1, 2, 3]
c = list(a)          # a real copy (a[:] or a.copy() also work)
c.append(99)

print("a =", a)      # untouched this time
print("c =", c)
print("same object?", a is c)

Rule of thumb: `=` never copies data, it only attaches a name. The same applies when you pass a list into a function — the function receives a label to *your* list and can modify it. That is sometimes exactly what you want, and sometimes a nasty surprise; either way, now you can predict it.

## Dicts: labelled data

A **dict** (dictionary) maps **keys** to **values** — like a phone book maps names to numbers. Where a list answers "what is at position 3?", a dict answers "what is stored under this key?" — and it does so almost instantly, no matter how big it grows.

In [ ]:
extensions = {"Ana": 4117, "Ben": 4203, "Chi": 4111}

print(extensions["Ana"])              # look up by key
extensions["Dmitri"] = 4300           # add a new entry
extensions["Ana"] = 4118              # overwrite an existing one

print(extensions.get("Zoe", 0))       # .get avoids a KeyError: default if missing
print("Ben" in extensions)            # membership tests look at KEYS

for name, ext in extensions.items():  # loop over key-value pairs
    print(f"  {name:>7} -> x{ext}")

The workhorse dict pattern is **counting**: use each item as a key and keep a tally as the value. `.get(key, 0)` makes this a one-liner per item — "current count, or 0 if we haven't seen it yet, plus one".

In [ ]:
lyric = "row row row your boat gently down the stream"

counts = {}
for word in lyric.split():            # .split() chops on whitespace
    counts[word] = counts.get(word, 0) + 1

# Show the tallies, most frequent first:
for word in sorted(counts, key=counts.get, reverse=True):
    print(f"{word:>7}: {'#' * counts[word]}  ({counts[word]})")

Three `row`s, one of everything else — a word frequency table in four lines. This exact pattern (with bigger data) powers search engines, spam filters, and your keyboard's autocomplete.

## Sets: membership without duplicates

A **set** is an unordered bag of *unique* items. Adding a duplicate is a no-op. Sets shine at two jobs: removing duplicates, and lightning-fast "have we seen this?" checks, plus set algebra (union, intersection, difference) straight out of math class.

In [ ]:
signups = ["ana", "ben", "ana", "chi", "ben", "ana"]
unique = set(signups)
print(unique, "-", len(unique), "distinct people")

chess_club = {"ana", "ben", "chi"}
robotics_club = {"ben", "dmitri", "chi"}

print("in either club:  ", chess_club | robotics_club)    # union
print("in both clubs:   ", chess_club & robotics_club)    # intersection
print("chess only:      ", chess_club - robotics_club)    # difference

Note the curly-brace printout has no reliable order — sets trade order away for speed and uniqueness.

## Tuples: fixed bundles

A **tuple** is like a list that can never change (**immutable**): perfect for small fixed bundles such as an (x, y) coordinate. Because they can't change, tuples may serve as dict keys — lists may not. Python also lets you **unpack** a tuple into separate names in one line.

In [ ]:
point = (3, 7)
x, y = point                      # unpacking
print(f"x={x}, y={y}")

x, y = y, x                       # the famous one-line swap
print(f"after swap: x={x}, y={y}")

# Tuples as dict keys: a tiny treasure map grid
treasure = {(0, 0): "start", (2, 3): "gold!", (4, 1): "trap"}
print(treasure[(2, 3)])

### Choosing a collection

| You need... | Use |
|---|---|
| an ordered sequence you'll modify | `list` |
| key → value lookup | `dict` |
| uniqueness / fast membership / set algebra | `set` |
| a small fixed bundle (or a dict key) | `tuple` |

## Classes: inventing your own types

Collections organise data, but the *rules* about that data still float around loose in your code. A **class** bundles data (**attributes**) and behaviour (**methods**) into a new type of your own. Each value stamped out from the class is an **object** (or *instance*).

Let's start with tradition — a `Dog`:

In [ ]:
class Dog:
    def __init__(self, name, age):    # runs when a new Dog is created
        self.name = name              # attribute: data stored ON this dog
        self.age = age

    def bark(self):                   # method: behaviour of this dog
        return f"{self.name} says: Woof!"

rex = Dog("Rex", 3)
luna = Dog("Luna", 1)
print(rex.bark())
print(luna.bark())
print(f"{rex.name} is {rex.age}; {luna.name} is {luna.age}")

Unpacking the magic words:

- `__init__` is the **constructor** — Python calls it automatically in `Dog("Rex", 3)` to set up the new object;
- `self` is the object being operated on. When you write `rex.bark()`, Python quietly passes `rex` in as `self` — that's how the method knows *whose* name to use;
- `rex` and `luna` are two separate objects: same blueprint, independent data.

Objects keep their own state, and methods can *change* that state:

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age
        self.tricks = []                       # every dog starts untrained

    def bark(self):
        return f"{self.name} says: Woof!"

    def learn(self, trick):
        self.tricks.append(trick)

    def show_off(self):
        if not self.tricks:
            return f"{self.name} knows no tricks yet."
        return f"{self.name} can: {', '.join(self.tricks)}"

rex = Dog("Rex", 3)
rex.learn("sit")
rex.learn("roll over")
print(rex.show_off())
print(Dog("Luna", 1).show_off())               # a brand-new dog: still untrained

Notice `self.tricks` is created inside `__init__`, so *each* dog gets its own fresh list — Luna's emptiness is untouched by Rex's education.

## A useful class: `WeatherStation`

Toy dogs aside, here is the same idea doing real work: a weather station that *accumulates* readings and can summarise them. The data (`self.readings`) and the calculations that belong to it (min/max/average) finally live in one place.

In [ ]:
class WeatherStation:
    def __init__(self, name):
        self.name = name
        self.readings = []

    def record(self, temp):
        self.readings.append(temp)

    def minimum(self):
        return min(self.readings)

    def maximum(self):
        return max(self.readings)

    def average(self):
        return sum(self.readings) / len(self.readings)

Defining a class produces no output — it's a blueprint, sitting ready. Now let's stamp out a station and feed it a week of temperatures:

In [ ]:
station = WeatherStation("Rooftop")
for temp in [18.5, 21.0, 19.2, 23.8, 17.1, 20.4, 22.9]:
    station.record(temp)

print(f"Station: {station.name} ({len(station.readings)} readings)")
print(f"  min: {station.minimum():.1f}")
print(f"  max: {station.maximum():.1f}")
print(f"  avg: {station.average():.2f}")

The caller never touches the list directly — it just asks the station questions. If we later store readings differently (a file, a database), the callers don't change at all. That separation is the real payoff of classes.

## Composition: objects inside objects

Classes get powerful when objects *contain* other objects — called **composition**, or the "has-a" relationship. A dog house *has a* dog (or stands empty). We model that by storing a `Dog` object (or `None` — Python's "nothing here" value) as an attribute:

In [ ]:
class DogHouse:
    def __init__(self, colour):
        self.colour = colour
        self.occupant = None                  # empty at first

    def move_in(self, dog):
        self.occupant = dog

    def describe(self):
        if self.occupant is None:
            return f"An empty {self.colour} dog house."
        return f"A {self.colour} dog house - inside: {self.occupant.bark()}"

kennel = DogHouse("red")
print(kennel.describe())
kennel.move_in(Dog("Rex", 3))
print(kennel.describe())

Look at `self.occupant.bark()` — the house doesn't know *how* to bark; it delegates to the dog it contains. Chains of objects asking each other for help is how large programs are actually structured. (The other classic relationship, inheritance — "is-a" — you'll meet later; composition is the one to reach for first.)

## Mini project: a gradebook

Time to combine everything: a `Student` **class** holds one student's scores and computes their average; a **dict** maps names to `Student` objects for instant lookup. Dict + class is a hugely common pairing.

In [ ]:
class Student:
    def __init__(self, name):
        self.name = name
        self.scores = []

    def add_score(self, score):
        self.scores.append(score)

    def average(self):
        if not self.scores:          # guard: no scores yet
            return 0.0
        return sum(self.scores) / len(self.scores)

The `if not self.scores` guard protects `average()` from dividing by zero on a student with no scores yet — defensive habits like this prevent real crashes. Now the gradebook itself:

In [ ]:
gradebook = {}
for name in ["Ana", "Ben", "Chi"]:
    gradebook[name] = Student(name)

results = [("Ana", 92), ("Ben", 71), ("Ana", 88), ("Chi", 85),
           ("Ben", 79), ("Chi", 91), ("Ana", 95)]
for name, score in results:
    gradebook[name].add_score(score)

print("name  | scores           | average")
print("------+------------------+--------")
for name, student in gradebook.items():
    print(f"{name:<5} | {str(student.scores):<16} | {student.average():>6.2f}")

Trace one line: `gradebook[name]` uses the **dict** to fetch the right `Student` **object**, then `.add_score(score)` asks that object to update itself. Each layer does its own job — the dict finds, the class computes.

## What you just learned

- Lists hold ordered mutable data — and `=` makes an **alias**, not a copy (`list(a)` copies).
- Dicts map keys to values; `.get(key, 0) + 1` is the counting idiom.
- Sets enforce uniqueness and do fast membership plus union/intersection.
- Tuples are immutable bundles that unpack neatly and can be dict keys.
- Classes bundle attributes and methods; `__init__` builds objects; `self` is "this object".
- Composition ("has-a") nests objects; dict + class organises real programs.

## Try it yourself

Each scaffold below runs as-is — fill in the `# your code here` parts and un-comment the tests.

### Exercise 1 — Dedupe, keep order

`set()` removes duplicates but scrambles order. Write `dedupe(items)` that returns a new list with duplicates removed but the *first-seen order kept*: walk the list, track what you've seen in a set, and append only new items to the result.

In [ ]:
def dedupe(items):
    seen = set()
    result = []
    # your code here
    return result

# Uncomment to test:
# assert dedupe(["b", "a", "b", "c", "a"]) == ["b", "a", "c"]
# assert dedupe([1, 1, 1]) == [1]
# assert dedupe([]) == []
# print("dedupe works!")

### Exercise 2 — Invert a dict

Write `invert(d)` that swaps keys and values: `{"Ana": 4117}` becomes `{4117: "Ana"}`. Loop over `d.items()`. (Assume the values are unique — think about *why* that assumption is necessary!)

In [ ]:
def invert(d):
    # your code here
    pass

# Uncomment to test:
# assert invert({"Ana": 4117, "Ben": 4203}) == {4117: "Ana", 4203: "Ben"}
# assert invert({}) == {}
# print("invert works!")

### Exercise 3 — A `BankAccount` class

Write a `BankAccount` class: `__init__(self, owner)` starts the balance at `0.0`; `deposit(amount)` adds to it; `withdraw(amount)` subtracts *only if* enough money is there and returns `True`/`False` for success/failure.

In [ ]:
class BankAccount:
    def __init__(self, owner):
        self.owner = owner
        self.balance = 0.0

    def deposit(self, amount):
        pass    # your code here

    def withdraw(self, amount):
        pass    # your code here: return True or False

# Uncomment to test:
# acct = BankAccount("Ana")
# acct.deposit(100.0)
# assert acct.withdraw(30.0) == True and acct.balance == 70.0
# assert acct.withdraw(500.0) == False and acct.balance == 70.0
# print("BankAccount works!")

### Exercise 4 — Temperature spread

Write a function `spread(station)` that takes a `WeatherStation` object and returns the difference between its hottest and coldest readings. Use the *methods* the station already offers rather than reaching into `station.readings` yourself.

In [ ]:
def spread(station):
    # your code here
    pass

test_station = WeatherStation("Valley")
for t in [12.0, 19.5, 8.5, 15.0]:
    test_station.record(t)

# Uncomment to test:
# assert spread(test_station) == 11.0     # 19.5 - 8.5
# print("spread works!")

### Exercise 5 — Letter grades for the gradebook

Write `letter_grade(average)` returning `"A"` for 90+, `"B"` for 80–89, `"C"` for 70–79, and `"F"` below 70. Then (second step) loop over the `gradebook` dict from earlier and print each student's name, average, and letter.

In [ ]:
def letter_grade(average):
    # your code here
    pass

# Uncomment to test the function...
# assert letter_grade(95) == "A"
# assert letter_grade(85) == "B"
# assert letter_grade(70) == "C"
# assert letter_grade(69.9) == "F"
# ...then produce the report:
# for name, student in gradebook.items():
#     print(name, round(student.average(), 1), letter_grade(student.average()))